# Taller Práctico 01 — Dataset A: Retail (panadería, Medellín)

In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

df_limpio = pd.read_csv("../data/raw/retail_ventas_LIMPIO.csv")
df_contaminado = pd.read_csv("../data/raw/retail_ventas_CONTAMINADO.csv")

print("LIMPIO:", df_limpio.shape)
print("CONTAMINADO:", df_contaminado.shape)

df_limpio.info()
df_limpio.head()

LIMPIO: (420, 11)
CONTAMINADO: (430, 11)
<class 'pandas.DataFrame'>
RangeIndex: 420 entries, 0 to 419
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   transaction_id   420 non-null    str    
 1   store_id         420 non-null    str    
 2   comuna           420 non-null    str    
 3   date             420 non-null    str    
 4   category         420 non-null    str    
 5   units_sold       420 non-null    int64  
 6   unit_price       420 non-null    float64
 7   payment_method   420 non-null    str    
 8   customer_rating  420 non-null    int64  
 9   store_lat        420 non-null    float64
 10  store_lon        420 non-null    float64
dtypes: float64(3), int64(2), str(6)
memory usage: 36.2 KB


,transaction_id,store_id,comuna,date,category,units_sold,unit_price,payment_method,customer_rating,store_lat,store_lon
0,T00001,S02,Laureles,2025-05-09,Bebidas,2,5100.0,App,5,6.24625,-75.58917
1,T00002,S02,Laureles,2025-06-03,Pasteles,4,8900.0,App,3,6.24605,-75.58977
2,T00003,S07,Aranjuez,2025-05-31,Snacks,1,3300.0,Tarjeta,4,6.27391,-75.55584
3,T00004,S04,Robledo,2025-06-07,Pan,3,3200.0,Efectivo,5,6.27054,-75.59354
4,T00005,S02,Laureles,2025-01-26,Bebidas,5,2100.0,Tarjeta,3,6.24270,-75.58825


## Parte 2 — Análisis y transformación de datos con problemas

### 2.1 Diagnóstico de calidad (GIGO)

1. **Valores faltantes (completitud):** cuatro columnas tienen nulos — `date` (10),
   `units_sold` (21), `unit_price` (12), `customer_rating` (64).
   Detección: `df.isnull().sum()`

2. **Duplicados de evento de negocio (unicidad):** 10 `transaction_id` repetidos,
   además de 1 duplicado exacto de fila.
   Detección: `df['transaction_id'].duplicated().sum()`

3. **Etiquetas categóricas inconsistentes (consistencia):** `category` tiene 19
   valores únicos para 4 categorías reales (mayúsculas, espacios, sinónimos).
   Detección: `df['category'].unique()`

4. **Formatos de fecha mixtos (consistencia):** `date` mezcla ISO, US y texto largo.
   Detección: inspección manual con `df['date'].sample(15)`

5. **Valores imposibles (validez):** `units_sold` con mínimo -6; `customer_rating`
   con valores -1 y 7, fuera de la escala 1-5.
   Detección: `df['units_sold'].describe()`, `df['customer_rating'].describe()`

In [6]:
diagnostico_gigo = pd.DataFrame({
    'problema': [
        'Valores faltantes',
        'Duplicados de evento de negocio',
        'Etiquetas categóricas inconsistentes',
        'Formatos de fecha mixtos',
        'Valores imposibles'
    ],
    'pilar_gigo': ['Completitud', 'Unicidad', 'Consistencia', 'Consistencia', 'Validez'],
    'evidencia': [
        'date:10, units_sold:21, unit_price:12, customer_rating:64 nulos',
        '10 transaction_id repetidos + 1 fila duplicada exacta',
        "19 valores únicos en 'category' para 4 categorías reales",
        "date mezcla ISO, US ('06-24-2025') y texto ('May 21, 2025')",
        'units_sold min=-6; customer_rating con -1 y 7 (fuera de 1-5)'
    ],
    'metodo_deteccion': [
        "df.isnull().sum()",
        "df['transaction_id'].duplicated().sum()",
        "df['category'].unique()",
        "df['date'].sample(15)",
        "df['units_sold'].describe()"
    ]
})

diagnostico_gigo.to_csv('../results/tabla_diagnostico_gigo.csv', index=False)
diagnostico_gigo

,problema,pilar_gigo,evidencia,metodo_deteccion
0,Valores faltantes,Completitud,"date:10, units_sold:21, unit_price:12, custome...",df.isnull().sum()
1,Duplicados de evento de negocio,Unicidad,10 transaction_id repetidos + 1 fila duplicada...,df['transaction_id'].duplicated().sum()
2,Etiquetas categóricas inconsistentes,Consistencia,19 valores únicos en 'category' para 4 categor...,df['category'].unique()
3,Formatos de fecha mixtos,Consistencia,"date mezcla ISO, US ('06-24-2025') y texto ('M...",df['date'].sample(15)
4,Valores imposibles,Validez,units_sold min=-6; customer_rating con -1 y 7 ...,df['units_sold'].describe()


### 2.2 Fechas

In [7]:
df_contaminado['date'] = pd.to_datetime(df_contaminado['date'], errors='coerce', format='mixed')

n_nulos_fecha = df_contaminado['date'].isnull().sum()
pct_nulos_fecha = round(n_nulos_fecha / len(df_contaminado) * 100, 1)
print(f"{n_nulos_fecha} fechas no convertibles de {len(df_contaminado)} ({pct_nulos_fecha}%)")

10 fechas no convertibles de 430 (2.3%)


En nuestro caso real, el 2.3% de las fechas (10 de 430) no logran convertirse
(el 8% es el escenario que plantea el enunciado para razonar en general).

**Decisión: dejarlas como nulo explícito (NaT), no eliminarlas ni imputarlas.**

- **No eliminar:** la fila conserva información válida en otras columnas
  (`category`, `unit_price`, `store_id`) que sigue siendo útil para el análisis
  por categoría o por tienda, aunque no se sepa la fecha exacta.
- **No imputar:** no existe una forma razonable de inferir la fecha de una
  transacción a partir de las demás columnas — imputar (ej. con la moda o
  una fecha promedio) introduciría un dato falso que distorsionaría cualquier
  análisis de tendencia temporal (ventas por mes, estacionalidad).
- **Nulo explícito es la opción honesta:** cualquier análisis que agrupe por
  fecha (`groupby` por mes, gráficas de series de tiempo) excluye automáticamente
  esas filas sin necesidad de una regla adicional, y sin falsear el resto de la serie.

Esta decisión cambiaría si el porcentaje de fechas faltantes fuera mucho más alto
(ej. 30-40%): ahí sí valdría la pena investigar si hay un patrón sistemático detrás
(ej. una tienda que nunca registra fecha) en vez de solo dejarlas como nulas.

### 2.3 Variable categórica

In [16]:
df_contaminado['category'] = (
    df_contaminado['category']
    .str.strip()
    .str.lower()
    .str.rstrip('s')  # unifica singular/plural donde aplica
)

# Mapeo explícito para los casos que no se resuelven solo con strip/lower
mapeo_categorias = {
    'snack': 'Snacks', 'snak': 'Snacks',
    'pan': 'Pan', 'pane': 'Pan',
    'pastele': 'Pasteles', 'pastel': 'Pasteles',
    'bebida': 'Bebidas'
}
df_contaminado['category'] = df_contaminado['category'].map(mapeo_categorias)

print(df_contaminado['category'].value_counts())
print(f"Suma total: {df_contaminado['category'].notna().sum()}")

category
Snacks      117
Pan         112
Bebidas     107
Pasteles     94
Name: count, dtype: int64
Suma total: 430


**Criterio para "misma categoría":** se normalizó primero el texto con
`.str.strip()` (elimina espacios accidentales como `' Pasteles'`), `.str.lower()`
(unifica mayúsculas: `'PASTELES'`, `'Pasteles'`), y `.str.rstrip('s')` (unifica
singular/plural: `'Pastel'` vs `'Pasteles'`). Esto redujo 19 valores a 7 variantes.

Las variantes restantes (`'snak'` por error de tipeo, `'pane'` en vez de `'pan'`,
`'pastele'` en vez de `'pastel'`) no se resuelven con transformaciones automáticas
porque son errores de escritura, no de formato — por eso se corrigieron con un
mapeo explícito basado en similitud semántica y fonética con las 4 categorías
válidas del diccionario de variables (Anexo A): Bebidas, Pan, Pasteles, Snacks.

El riesgo de automatizar demasiado (ej. usar distancia de Levenshtein sin revisión
humana) es fusionar por error categorías que en el negocio real son distintas.
Con solo 19 variantes, la revisión manual es más confiable que una regla genérica.

### 2.4 Georreferenciación

In [17]:
LAT_MIN, LAT_MAX = 5.9, 6.6
LON_MIN, LON_MAX = -75.75, -75.35

coord_invalida = ~(
    df_contaminado['store_lat'].between(LAT_MIN, LAT_MAX) &
    df_contaminado['store_lon'].between(LON_MIN, LON_MAX)
)
df_contaminado['coord_valida'] = ~coord_invalida

print(f"{coord_invalida.sum()} filas con coordenadas inválidas de {len(df_contaminado)}")
df_contaminado.loc[coord_invalida, ['store_id', 'store_lat', 'store_lon']]

6 filas con coordenadas inválidas de 430


,store_id,store_lat,store_lon
125,S05,0.00000,0.00000
146,S06,0.00000,0.00000
304,S04,0.00000,0.00000
374,S01,46.20987,-75.56624
399,S02,46.24496,-75.58880
413,S04,46.27285,-75.59422


**Regla de validación:** Medellín se ubica aproximadamente entre lat 5.9°–6.6°N
y lon -75.75°–-75.35°O. Cualquier coordenada fuera de ese rango se marca como
inválida (`coord_valida = False`) en vez de eliminarse, para no perder el resto
de la información de la transacción.

**¿Corregir automáticamente un swap lat/lon o marcar para revisión manual?**

Se opta por **marcar para revisión manual, no corregir automáticamente**.

- **Riesgo de corregir automáticamente (swap):** asumir que el único error
  posible es un intercambio de columnas. Si el error real es otro (dígito
  mal digitado, cero de más, dato corrupto), el swap automático generaría
  una coordenada "válida" pero completamente falsa, lo cual es peor que un
  nulo explícito porque parece confiable sin serlo.
- **Riesgo de marcar para revisión manual:** requiere tiempo humano y retrasa
  el análisis geoespacial. Con solo 6 filas afectadas de 430, este costo es
  bajo y el beneficio de no introducir ubicaciones falsas lo compensa.

**Nota sobre el patrón real encontrado:** de las 6 coordenadas inválidas,
3 corresponden a pares `(0, 0)` — un placeholder típico de dato faltante
mal codificado, no una ubicación real — y las otras 3 tienen una latitud
cercana a 46° (ej. `46.20987`), que no es un swap con la longitud sino un
posible error de tipeo (un dígito de más al inicio; el valor real
probablemente es `6.20987`, coherente con Medellín).

Esto refuerza la decisión de no corregir automáticamente: una regla de
"swap lat/lon" no habría arreglado ninguno de los 6 casos reales, porque
ninguno es un swap genuino. Aplicar una corrección automática basada en un
supuesto incorrecto habría generado ubicaciones igual de falsas que el
problema original.

Dado que el propósito del análisis (Parte 3) es identificar qué tienda
requiere intervención, una ubicación falsa podría llevar a una recomendación
de negocio incorrecta — por eso se prioriza la certeza sobre la automatización.

### 2.5 Imputación y valores imposibles

In [18]:
# a) Rango válido: no pueden existir unidades vendidas negativas o en cero
#    (una transacción registrada implica al menos 1 unidad vendida)
valores_imposibles = df_contaminado['units_sold'] <= 0
print(f"{valores_imposibles.sum()} filas con units_sold imposible (<=0)")
print(df_contaminado.loc[valores_imposibles, ['transaction_id', 'units_sold']])

# b) Estrategia: imputar con la mediana de la categoría correspondiente
#    (más robusta que la media ante outliers, y respeta que categorías
#    distintas tienen patrones de venta distintos)
mediana_por_categoria = df_contaminado.groupby('category')['units_sold'].transform('median')
df_contaminado.loc[valores_imposibles, 'units_sold'] = mediana_por_categoria[valores_imposibles]

8 filas con units_sold imposible (<=0)
    transaction_id  units_sold
2           T00419        -2.0
111         T00368        -3.0
120         T00088        -4.0
123         T00322        -6.0
125         T00011        -4.0
188         T00304        -2.0
190         T00139        -3.0
319         T00267        -5.0


**a) Rango válido:** `units_sold` debe ser un entero positivo (≥1). Un valor
negativo o cero no tiene sentido físico: si existe una transacción registrada,
por definición se vendió al menos una unidad.

**b) Estrategia — imputar con la mediana por categoría, no eliminar:**
Se eligió imputar en vez de eliminar porque estas filas conservan información
válida en otras columnas (`unit_price`, `category`, `store_id`, `date`) que
sigue siendo útil para el análisis. Se usa la **mediana** (no la media) porque
es más robusta ante outliers, y se calcula **por categoría** (no global)
porque categorías distintas (ej. Pasteles vs. Snacks) tienen patrones de venta
distintos — usar una sola mediana global ignoraría esa heterogeneidad.

**c) Sesgo si la estrategia es incorrecta:** si en vez de imputar por categoría
se hubiera usado la media global, se subestimaría sistemáticamente las
categorías de alta rotación (ej. Pan) y se sobrestimaría las de baja rotación
(ej. Pasteles), distorsionando la comparación de rotación por categoría que
es clave para la decisión de inventario en la Parte 3. Si en cambio se hubiera
optado por eliminar estas filas, se perdería información de precio/tienda de
esas transacciones sin necesidad, reduciendo el tamaño de muestra sin
justificación.

### 2.6 Duplicados y llave de negocio

**Llave de negocio propuesta:** la combinación de `store_id` + `date` + `category`
+ `unit_price` (no `transaction_id` solo).

**Por qué `transaction_id` no basta como llave de negocio:**
`transaction_id` es un identificador técnico generado al momento del registro —
si el sistema de captura genera un ID nuevo cada vez que se reintenta guardar
una transacción (por ejemplo, ante un error de red o un doble clic en el punto
de venta), dos filas con `transaction_id` distintos pueden representar el
**mismo evento de negocio real** duplicado. En ese caso, buscar duplicados
exactos de fila (`df.duplicated()`) o incluso duplicados de `transaction_id`
no detecta este tipo de duplicado — hay que compararlo contra atributos que
describen el evento en sí: misma tienda, misma fecha, misma categoría y mismo
precio unitario apuntan a que probablemente es la misma venta registrada dos veces.

**Por qué un duplicado exacto de fila no es lo mismo que un duplicado de
evento de negocio:**
Un duplicado exacto de fila (mismos valores en *todas* las columnas, incluido
`transaction_id`) casi siempre indica un error técnico de carga de datos
(ej. el mismo CSV importado dos veces). Un duplicado de evento de negocio, en
cambio, puede tener `transaction_id` distintos pero describir la misma venta
real — y también puede darse el caso contrario: dos transacciones legítimas
y diferentes que coinciden en tienda, fecha, categoría y precio por pura
coincidencia (ej. dos clientes comprando lo mismo el mismo día), que un
análisis ingenuo de "duplicado de negocio" marcaría erróneamente como
duplicado sin serlo. Por eso esta llave debe usarse con criterio, no de forma
automática — idealmente cruzando también la hora exacta si estuviera disponible.